# Midterm: exercises (JEM207) 
--- 
Boris Gerát 23.11 

[Link](https://drive.google.com/file/d/1YmC18z3kMihKM5fVY9H9Y9vQO0r-ZAnP/view?usp=sharing) for folder

Fill in the empty code cells. Hints are below each task; try without them first.

- 25 points in total: Part 1 (14 pt), Part 2 (9 pt), Part 3 (2 pt), plus a bonus problem (up to 3 pt).
- Solve each task in its own cell.
- The notebook must run from scratch (Restart kernel → Run all) without errors.

In [ ]:
# Install the required packages if they are missing
import importlib.util, subprocess, sys

for pkg in ['pandas', 'numpy', 'matplotlib']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

In [ ]:
import pandas as pd
import numpy as np
import zipfile
from datetime import datetime
import matplotlib.pyplot as plt
import random

### Part 1 (14 points)

#### Get the data (0 points)

The zip file contains one CSV per company. Each file name is the company ticker (do not rename the files).

- Load every CSV from the zip.
- Add a `Ticker` column with the ticker taken from the file name.
- Make sure `Date` is parsed as a datetime, not as text.
- Stack all companies into one DataFrame `df` with `Date` as the index.

Use the `Close` series for all tasks below unless a task says otherwise.

In [ ]:
zip_file = zipfile.ZipFile('42459015.zip')
zip_file.namelist()

In [ ]:
list_of_dfs = []
for file_name in zip_file.namelist():
    if not file_name.endswith('.csv'):
        continue
    ticker = file_name.split('/')[-1].replace('.csv', '')
    one_company = pd.read_csv(zip_file.open(file_name), parse_dates=['Date'])
    one_company['Ticker'] = ticker
    list_of_dfs.append(one_company)

df = pd.concat(list_of_dfs).set_index('Date')
tickers = df['Ticker'].unique()
print(tickers, df.index.dtype)
df.head()

#### 1. (2 pt) High vs. Low
- Is there a company whose `High` and `Low` are equal (no difference) on at least one trading day?
- What does `High == Low` mean for the stock's trading behaviour on such a day?

> **Hint:** Compare `High == Low` within each company; `.any()` says if it happens, `.sum()` on how many days. Look at `Volume` on such days.

*Your answer:* 

#### 2. (1 pt) What is the highest and the lowest price each company recorded?

Choose the column(s) wisely: which column can contain the highest price of a day, and which one the lowest?

> **Hint:** The daily maximum is in `High`, the minimum in `Low`. Use `groupby('Ticker').agg(...)` with `max` and `min`.

#### 3. (1 pt) Log returns
- Calculate daily logarithmic returns from `Close`: $r_t = \log(P_t) - \log(P_{t-1})$.
- For each company, report these statistics of the returns:
  - min
  - max
  - mean
  - standard deviation
  - skewness

> **Hint:** `np.log(df['Close'])`, then `.diff()` within each company (`df.groupby('Ticker')`), otherwise the first day of one company is compared with the last day of another. Then `.agg(['min', 'max', 'mean', 'std', 'skew'])`.

#### 4. (2 pt) Second highest gain and loss
For each company, find the **date** of:
- the second highest gain (2nd largest log return),
- the second highest loss (2nd smallest, i.e. 2nd most negative, log return).

> **Hint:** Per company: `.dropna().sort_values()`; then `.index[1]` is the 2nd highest loss and `.index[-2]` the 2nd highest gain. Or: drop the `idxmax()` and call `idxmax()` again.

#### 5. (2 pt) Median calendar weekly volume
- For each company, compute the weekly volume: the sum of `Volume` over each calendar week.
- Report the median of these weekly sums for each company.

> **Hint:** `df.groupby('Ticker')['Volume'].resample('W', level='Date').sum()`, then the median per company with `.groupby(level='Ticker')`.

#### 6. (2 pt) Which company recorded the lowest total logarithmic return over the whole period?

Total log return from day 0 to day T: $r_T = \log(P_T) - \log(P_0) = \sum_{t=1}^{T} r_t$

> **Hint:** Log returns add up, so sum `ret` per company; `.idxmin()` gives the ticker.

#### 7. (2 pt) Volume classes
- Create a new column `volume_class` with the categories `"Low"`, `"Medium"`, `"High"`.
- Assign the categories by volume tertiles computed **within each company**, not across all companies together.

> **Hint:** `df.groupby('Ticker')['Volume'].rank(pct=True)` gives each day's position (0–1) within its company; cut it with `pd.cut(..., bins=[0, 1/3, 2/3, 1], labels=['Low', 'Medium', 'High'])`.

#### 8. (2 pt) Plots of log returns
- **(A)** Plot the log returns of the companies over time, ideally all in the same plot.
- **(B)** Plot the distribution of the log returns, ideally all in the same plot.

> **Hint:** (A) `df.pivot(columns='Ticker', values='ret')` gives one column per company; call `.plot()`. (B) `.plot.hist(bins=100, alpha=0.4, density=True)` on the same table (or `.plot.kde()`).

### Part 2 (9 points)

#### Load the data

Load the dataset of S&P 500 companies, `companies_no_subindustry.csv`.

In [ ]:
companies = pd.read_csv('companies_no_subindustry.csv')
companies.head()

#### 1. (1 pt) How many companies have a missing inclusion date (`included` column)?

> **Hint:** `.isna().sum()` on the `included` column only.

#### 2. (2 pt) Inclusion dates and company age
First remove the companies with a missing inclusion date. Then:
- Find the oldest and the youngest constituent in terms of **inclusion date** (`included`).
- Compute the median age of a constituent in years: age = today's year − year in `Founded`.
- `Founded` comes in different formats (e.g. `'1902'` vs `'2013 (1888)'`); unify it by taking the first 4 characters of each entry.

> **Hint:** `dropna(subset=['included'])`. Convert dates with `pd.to_datetime(..., dayfirst=True, errors='coerce')`; text dates do not sort correctly. `idxmin()` / `idxmax()` give the row label → `.loc[row, 'Security']`. Age: `Founded.str[:4]` → number → `datetime.today().year - year` → `.median()`.

#### 3. (2 pt) Sectors
- Compute the number of companies in each GICS sector.
- Visualize the sector proportions (e.g., pie chart).

> **Hint:** `value_counts()` on `GICS_Sector`, then `.plot.pie(autopct='%1.1f%%')`.

#### 4. (2 pt) Headquarters state
- Parse the `hq` column to extract the state.
- Describe the distribution of states (e.g., bar plot).

> **Hint:** `.str.split(',').str[-1].str.strip()`, then `value_counts()` and `.plot.bar()`. Some entries are countries, not US states.

#### 5. (2 pt) Merge with subindustries
- Load `companies_subindustry.csv` and merge it with the companies table using an appropriate key.
- Within the `"Health Care"` GICS sector, report the subindustry distribution (e.g., bar plot).

> **Hint:** The shared key is `Symbol`: `companies.merge(companies2, on='Symbol', how='left')`. Keep `GICS_Sector == 'Health Care'`, then `value_counts()` on `subindustry` and `.plot.barh()`.

### Part 3 (2 points)

#### 1. (1 pt) Generate four random variables
- Each variable comes from a different distribution (e.g., Normal, Uniform, Binomial, Exponential).
- Each variable has its own random number of observations between 100 and 500.

> **Hint:** Draw a new `random.randint(100, 500)` for each variable and pass it as `size=`. The lengths differ, so keep the variables in a dict, not a DataFrame.

#### 2. (1 pt) Draw a 2×2 grid of plots visualizing the random variables
- One panel per random variable.
- Use at least 2 of these plot types: PDF, CDF, histogram, boxplot.

> **Hint:** `fig, axes = plt.subplots(2, 2)` and loop over `axes.flat`. E.g. `ax.hist(...)`, `ax.boxplot(...)`, empirical CDF: `ax.plot(np.sort(x), np.arange(1, n + 1) / n)`.

### Bonus problem (up to 3 points)

A positive integer $d$ is a divisor of an integer $n$ if $n$ is divisible by $d$ without a remainder. For example, the divisors of 12, excluding 1 and 12 itself, are 2, 3, 4, and 6. A prime number has no divisors other than 1 and itself.

#### 1. (2 pt) List of lists of divisors
- For each integer from 2 to 20 (inclusive), find all its divisors, excluding 1 and the number itself.
- Each inner list holds the divisors of one integer, in ascending order.
- For prime numbers the inner list is empty.
- Collect the inner lists in one list (the first entry belongs to 2, the last to 20).

> **Hint:** `d` is a divisor if `n % d == 0`; check `d` in `range(2, n)` for each `n` in `range(2, 21)`. Primes give `[]` automatically.

#### 2. (1 pt) Dictionary of divisors
- Build a dictionary where the keys are the integers from 2 to 20.
- The values are the corresponding lists of divisors from task 1.

> **Hint:** `dict(zip(range(2, 21), divisors))` or a dict comprehension.